In [ ]:
#use "../lib/nb.ml";;
load_css ()

# A Shift-Reduce Parser for Arithmetic Expressions

In this notebook we implement a generic *shift reduce parser*.  The parse table that we use 
implements the following grammar for arithmetic expressions:
$$
  \begin{eqnarray*}
  \mathrm{expr}        & \rightarrow & \mathrm{expr}\;\;\texttt{'+'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{expr}\;\;\texttt{'-'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{product}                                    \\[0.2cm]
  \mathrm{product}     & \rightarrow & \mathrm{product}\;\;\texttt{'*'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{product}\;\;\texttt{'/'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{factor}                                     \\[0.2cm]
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}     \\
                       & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

In [ ]:
load_lib "regex.ml";;
run_notebook "Parse-Table.ipynb"

The function `tokenize` transforms the string `s` into a list of pairs of the form `(type, value)`. See below for an
example.  The notebook `Parse-Table.ipynb`, which contains the parse table and the types of the parse table, has been
loaded in the previous cell.

In [ ]:
let tokenize (s : string) : (token * string) list =
  (* Transform the string s into a list of (type, value) pairs. *)
  let lexSpec = {|([ \t\n]+)      |  # blanks and tabs
                  ([1-9][0-9]*|0) |  # number
                  ([-+*/()])      |  # arithmetical operators
                  (.)                # unrecognized character
               |}
  in
  List.filter_map
    (function
      | [ ws; number; operator; error ] ->
          if ws <> "" then None
          else if number <> "" then Some ("NUMBER", number)   (* the generic type 'NUMBER' and the actual value *)
          else if operator <> "" then Some (operator, operator) (* for operators, type and value are identical *)
          else Some ("ERROR", error)
      | _ -> assert false)
    (Re.findall_groups ~flags:Re.[VERBOSE] lexSpec s)

In [ ]:
tokenize "1 + 2 * (3 - 4)"

The parser builds a *parse tree*.  In Python, a leaf of this tree is a string and an inner node is a pair consisting
of a variable and a tuple of children.  In OCaml, these are the constructors `Leaf` and `Node` of the type `parse_tree`.
The module `ShiftReduceParser` contains the record type `t`, which stores the three tables of the parser.

In [ ]:
type parse_tree = Leaf of string | Node of string * parse_tree list

module ShiftReduceParser = struct
  type t = {
    mActionTable : (state * token, action) Hashtbl.t;
    mGotoTable   : (state * variable, state) Hashtbl.t;
    mStateTable  : (state, marked_rule FSet.t) Hashtbl.t;
  }

  let make actionTable gotoTable stateTable : t =
    { mActionTable = actionTable; mGotoTable = gotoTable; mStateTable = stateTable }
end

The function `parse` extends the module `ShiftReduceParser`.  It prints the stacks in every step.  Besides the stacks of
states and symbols, it maintains the stack `forest`, which contains the parse trees of the symbols on the symbol stack.
The stacks are lists whose first element is the top of the stack, so they are reversed when they are printed.  The
result is `Some tree` if the parse is successful and `None` otherwise.

In [ ]:
module ShiftReduceParser = struct
  include ShiftReduceParser

  let rec drop (n : int) (l : 'a list) : 'a list = if n = 0 then l else drop (n - 1) (List.tl l)
  let rec take (n : int) (l : 'a list) : 'a list = if n = 0 then [] else List.hd l :: take (n - 1) (List.tl l)

  let parse (self : t) (tl : (token * string) list) : parse_tree option =
    let tl = Array.of_list (tl @ [ ("EOF", "EOF") ]) in
    let rec loop index (symbols : string list) (states : state list) (forest : parse_tree list) =
      let q = List.hd states in
      (* the current lookahead token: its type and its value *)
      let token_type, _token_value = tl.(index) in
      let remaining_tokens = List.map snd (Array.to_list (Array.sub tl index (Array.length tl - index))) in
      Printf.printf "States:  [ %s ]\n%!" (String.concat ", " (List.rev states));
      Printf.printf "Symbols: %s\n%!" (String.trim (String.concat " " (List.rev symbols @ [ "|" ] @ remaining_tokens)));
      (* use token_type for the table lookup *)
      match Hashtbl.find_opt self.mActionTable (q, token_type) with
      | None ->
          Printf.printf "Action(%s, %s) undefined.\n%!" q token_type;
          print_endline "Syntax error!\n";
          None
      | Some Accept ->
          print_endline "Accepting!\n";
          Some (List.hd forest)
      | Some (Shift s) ->
          Printf.printf "Shifting state %s\n\n%!" s;
          let _, token_value = tl.(index) in
          loop (index + 1) (token_type :: symbols) (s :: states) (Leaf token_value :: forest)
      | Some (Reduce (head, body)) ->
          Printf.printf "Reducing with rule %s → %s\n%!" head (String.concat " " body);
          let n        = List.length body in
          let children = List.rev (take n forest) in
          let forest   = Node (head, children) :: drop n forest in
          let symbols  = head :: drop n symbols in
          let states   = drop n states in
          let state    = List.hd states in
          loop index symbols (Hashtbl.find self.mGotoTable (state, head) :: states) forest
    in
    loop 0 [] [ "s0" ] []
end

In [ ]:
run_notebook "Parse-Table.ipynb"

## Visualization

In [ ]:
load_lib "graphviz.ml"

Visualize the given parse tree using `Graphviz`.  The local function `get_id` generates unique names for the nodes.
Since the counter is changed, it is a reference.

In [ ]:
let draw_tree (tree : parse_tree) : Graphviz.t =
  let dot = Graphviz.digraph "" in
  (* counter to generate unique IDs for each node in the graph *)
  let counter = ref 0 in
  let get_id () =
    incr counter;
    Printf.sprintf "node%d" !counter
  in
  let rec visit ?parent_id (node : parse_tree) : unit =
    (* create a unique ID for the current node *)
    let node_id = get_id () in
    match node with
    | Node (head, children) ->
        (* a non-terminal: draw the node (default shape is oval) *)
        Graphviz.node dot ~attrs:[ ("label", head) ] node_id;
        (* connect to the parent if it exists *)
        Option.iter (fun p -> Graphviz.edge dot p node_id) parent_id;
        (* recursively visit all children *)
        List.iter (visit ~parent_id:node_id) children
    | Leaf label ->
        (* a terminal: draw the node with a distinct shape and font *)
        Graphviz.node dot ~attrs:[ ("label", label); ("shape", "box"); ("fontname", "Courier") ] node_id;
        Option.iter (fun p -> Graphviz.edge dot p node_id) parent_id
  in
  (* start the traversal from the root *)
  visit tree;
  dot

## Testing

The function `test` returns `Some g`, where `g` is the graph of the parse tree, if the string can be parsed, and `None`
otherwise.  `Option.iter Graphviz.display` displays the graph if there is one.

In [ ]:
let test (s : string) : Graphviz.t option =
  let parser = ShiftReduceParser.make actionTable gotoTable stateTable in
  let tl     = tokenize s in
  Printf.printf "tokenlist: [%s]\n\n%!"
    (String.concat ", " (List.map (fun (t, v) -> Printf.sprintf "('%s', '%s')" t v) tl));
  match ShiftReduceParser.parse parser tl with
  | Some tree ->
      print_endline "Parse successful!";
      Some (draw_tree tree)
  | None ->
      print_endline "Parse failed!";
      None

In [ ]:
let pt = test "1 + 2 * 3"
;;
Option.iter Graphviz.display pt

In [ ]:
Option.iter Graphviz.display (test "1 + 2 * (3 - 4)")

In [ ]:
Option.iter Graphviz.display (test "1 + * 2")